# Day 2, Session 1 -- Demos: OpenAI API Engineering with Structured Outputs

This notebook contains all five instructor-led demos from Session 1. Run each cell, observe the output, and make sure you understand the pattern before moving on.

**Engineering context:** You are the engineer building production-grade API integrations. Consultants are your users -- they need reliable structured outputs, streaming for real-time briefings, and validated data pipelines they can trust.

In [ ]:
!pip install -q openai pydantic python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
from pydantic import BaseModel, Field
from typing import Optional, List

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demo 1: OpenAI API Deep Dive -- Streaming, Token Usage, and Finish Reasons

In production systems you need to: (a) track token usage for cost management, (b) stream responses for real-time briefings, and (c) inspect finish reasons to know if the model completed its analysis or was cut off.

**Scenario:** A partner asks the AI assistant to generate a concise market overview for a CEO briefing. We track tokens to manage costs across engagements and stream the response so the delivery team can review output in real time.

In [ ]:
# Demo 1 - OpenAI API Deep Dive

client = openai.OpenAI()

# Part A: Token usage tracking
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "Provide a 3-sentence executive summary of the global healthcare market outlook for a CEO briefing."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "150"))
)

print("=== Token Usage (Cost Tracking) ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nCEO Briefing Response:\n{response.choices[0].message.content}")

# Part B: Streaming responses
print("\n=== Streaming Response (Real-Time Review) ===")
stream = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "List the top 3 priorities for a digital transformation engagement at a Fortune 500 retailer, one per line."}],
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "100")),
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text: {collected_text}")

---
## Demo 2: Structured Output with JSON Mode -- Client Profile Extraction

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. This is critical for workflows where downstream systems must parse data reliably. You **must** include the word "JSON" in your prompt when using this mode.

**Scenario:** An engagement manager needs a structured client profile for a new healthcare engagement. JSON mode ensures the profile data is machine-readable for CRM and engagement tracking systems.

In [ ]:
# Demo 2 - Structured Output with JSON Mode

client = openai.OpenAI()

response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": "You are an engagement assistant that outputs JSON."},
        {"role": "user", "content": "Create a client profile for a new engagement with Acme Healthcare Corp. Include: client_name, industry, annual_revenue_usd, number_of_employees, headquarters, key_challenges (list), and engagement_type."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

# Parse and pretty-print
parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

# Access individual fields
print(f"\nClient      : {parsed.get('client_name', 'N/A')}")
print(f"Industry    : {parsed.get('industry', 'N/A')}")
print(f"Engagement  : {parsed.get('engagement_type', 'N/A')}")

---
## Demo 3: Function Calling -- Consulting Research Tools

Function calling lets the model decide **when** and **how** to call external tools. This enables an AI assistant to autonomously pull market data, run benchmarks, or look up competitor intelligence. The flow is:
1. You define tool schemas (e.g., `market_research`) and send them with the request
2. The model returns a `tool_calls` response (instead of regular content)
3. You execute the function locally (simulated data)
4. You send the result back to the model for a final answer

**Scenario:** An associate asks the AI for market research on a specific industry. The model decides to call the `market_research` tool to retrieve structured market data.

In [ ]:
# Demo 3 - Function Calling / Tool Use

client = openai.OpenAI()

# Step 1: Define the tools the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "market_research",
            "description": "Get market research data for a specific industry sector",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {
                        "type": "string",
                        "description": "The industry sector, e.g., 'healthcare', 'financial_services', 'energy'"
                    },
                    "region": {
                        "type": "string",
                        "enum": ["north_america", "europe", "asia_pacific", "global"],
                        "description": "Geographic region for the analysis"
                    }
                },
                "required": ["industry"]
            }
        }
    }
]

# Step 2: Simulated market research function
def market_research(industry, region="global"):
    """Simulated market research function returning industry data."""
    market_data = {
        "healthcare": {"market_size_usd_bn": 8450, "cagr_pct": 7.9, "key_trend": "AI-driven diagnostics and personalized medicine"},
        "financial_services": {"market_size_usd_bn": 28500, "cagr_pct": 6.2, "key_trend": "Embedded finance and real-time payments"},
        "energy": {"market_size_usd_bn": 6700, "cagr_pct": 8.5, "key_trend": "Energy transition and grid modernization"},
    }
    data = market_data.get(industry, {"market_size_usd_bn": 5000, "cagr_pct": 5.0, "key_trend": "Digital transformation"})
    return json.dumps({"industry": industry, "region": region, "market_size_usd_bn": data["market_size_usd_bn"], "cagr_pct": data["cagr_pct"], "key_trend": data["key_trend"]})

# Step 3: Make the initial API call
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[{"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

# Step 4: Check if the model wants to call a function
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"Function called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")

    # Step 5: Execute the function
    args = json.loads(tool_call.function.arguments)
    result = market_research(**args)
    print(f"Function result: {result}")

    # Step 6: Send the result back to the model
    follow_up = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "user", "content": "What does the healthcare market look like globally? I need data for a client engagement."},
            message,
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\nFinal response:\n{follow_up.choices[0].message.content}")
else:
    print(f"Direct response: {message.content}")

---
## Demo 4: Pydantic-Based Response Validation -- Engagement Summary

Pydantic models let you define the **exact shape** of the data you expect from the LLM, with type checking and constraints. If the LLM returns data that does not match, Pydantic will raise an error -- catching problems before they propagate through your workflow.

**Scenario:** Validate a structured engagement summary for a recently completed strategy engagement. The Pydantic model enforces that all required fields (client name, industry, engagement type, satisfaction score, etc.) are present and correctly typed.

In [ ]:
# Demo 4 - Pydantic-Based Response Validation

client = openai.OpenAI()

# Step 1: Define Pydantic models for structured output
class EngagementSummary(BaseModel):
    client_name: str = Field(description="Name of the client organization")
    industry: str = Field(description="Client industry sector")
    engagement_type: str = Field(description="Type of engagement, e.g., Strategy, M&A Due Diligence, Digital Transformation")
    duration_weeks: int = Field(description="Duration of the engagement in weeks")
    satisfaction_score: float = Field(ge=1.0, le=5.0, description="Client satisfaction score (1-5)")
    key_recommendation: str = Field(description="One-sentence primary recommendation")
    follow_on_opportunity: bool = Field(description="Whether there is a follow-on engagement opportunity")

# Step 2: Request structured output using JSON mode
response = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": (
            "You are an engagement tracking system. Output your engagement summary as JSON with these exact fields: "
            "client_name (string), industry (string), engagement_type (string), duration_weeks (integer), "
            "satisfaction_score (float 1-5), key_recommendation (string), follow_on_opportunity (boolean)."
        )},
        {"role": "user", "content": "Summarize the recently completed digital transformation engagement with Meridian Financial Group."}
    ],
    response_format={"type": "json_object"},
    max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "300"))
)

raw = response.choices[0].message.content
print("Raw JSON:")
print(raw)

# Step 3: Validate with Pydantic
try:
    summary = EngagementSummary.model_validate_json(raw)
    print("\nValidated EngagementSummary:")
    print(f"  Client          : {summary.client_name}")
    print(f"  Industry        : {summary.industry}")
    print(f"  Engagement Type : {summary.engagement_type}")
    print(f"  Duration        : {summary.duration_weeks} weeks")
    print(f"  Satisfaction    : {summary.satisfaction_score}/5.0")
    print(f"  Recommendation  : {summary.key_recommendation}")
    print(f"  Follow-on       : {'Yes' if summary.follow_on_opportunity else 'No'}")
except Exception as e:
    print(f"\nValidation error: {e}")

---
## Demo 5: Building a Structured Data Extraction Pipeline -- Engagement Team Extraction

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured team information from unstructured engagement notes. This pattern is the foundation for many agentic data-processing workflows -- from staffing databases to knowledge management systems.

**Scenario:** An engagement kickoff email contains team member details in unstructured text. We build a pipeline to extract each team member's name, role, email, practice area, and office location into structured data.

In [ ]:
# Demo 5 - Building a Structured Data Extraction Pipeline

client = openai.OpenAI()

class TeamMember(BaseModel):
    name: str = Field(description="Full name of the team member")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    role: Optional[str] = Field(default=None, description="Consulting role: Partner, Associate Partner, Engagement Manager, Associate, Business Analyst")
    practice: Optional[str] = Field(default=None, description="Practice area if found")

def extract_team_members(text: str) -> List[TeamMember]:
    """Extract structured team member information from unstructured engagement notes."""
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": (
                "Extract all consulting team member information from the text. "
                "Return a JSON object with a 'team_members' key containing a list. "
                "Each member should have: name, email (null if not found), "
                "phone (null if not found), role (null if not found), "
                "practice (null if not found)."
            )},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=int(os.getenv("DEFAULT_MAX_TOKENS", "500"))
    )

    data = json.loads(response.choices[0].message.content)
    members = []
    for item in data.get("team_members", []):
        try:
            members.append(TeamMember(**item))
        except Exception as e:
            print(f"Skipping invalid team member: {e}")
    return members

# Test the extraction pipeline
sample_text = """
Team, here is the staffing for the Apex Energy transformation engagement:

The engagement will be led by Partner Rajesh Gupta from our Houston office (Energy Practice).
Reach him at rajesh.gupta@mckinsey.com or (713) 555-0142.

Engagement Manager Sarah Okonkwo from the Chicago office will run day-to-day operations.
She is part of our Operations Practice. Her email is sarah.okonkwo@mckinsey.com.

We also have two Associates staffed: David Chen (Digital/McKinsey Digital, New York office,
david.chen@mckinsey.com) and Priya Sharma (Strategy & Corporate Finance, London office).

For analytics support, reach out to Business Analyst Tom Williams at tom.williams@mckinsey.com.
"""

print("Input text:")
print(sample_text)
print("=" * 60)
print("Extracted engagement team:")
print("=" * 60)

members = extract_team_members(sample_text)
for i, member in enumerate(members, 1):
    print(f"\nTeam Member {i}:")
    print(f"  Name     : {member.name}")
    print(f"  Email    : {member.email or 'N/A'}")
    print(f"  Phone    : {member.phone or 'N/A'}")
    print(f"  Role     : {member.role or 'N/A'}")
    print(f"  Practice : {member.practice or 'N/A'}")

---
## Summary

In this demo notebook you saw five production-grade OpenAI API patterns:

1. **Streaming and Token Tracking** -- Real-time output delivery and cost management.
2. **JSON Mode** -- Guaranteed valid JSON via `response_format={"type": "json_object"}`.
3. **Function Calling** -- Defining tools the model can invoke, handling the request-execute-respond loop.
4. **Pydantic Validation** -- Type-safe parsing and validation of LLM outputs.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable data extraction workflows.

**Next:** Open `D2S1_exercises.ipynb` to practice building a financial analysis tool with function calling.